In [3]:
import requests
import pandas as pd
import pandas as pd
from pyfaidx import Fasta # might need to do pip install first


base_url = "https://www.cbioportal.org/api"

# Choose study
profile = "crc_msk_2017_mutations"

# Choose sample ids 
sample_ids = [
"P-0003365-T01-IM5","P-0003575-T01-IM5","P-0003710-T01-IM5","P-0004159-T01-IM5",
"P-0004369-T01-IM5","P-0004502-T01-IM5","P-0004736-T01-IM5","P-0005446-T01-IM5",
"P-0005502-T01-IM5","P-0005569-T01-IM5","P-0005941-T01-IM5","P-0006136-T01-IM5",
"P-0006735-T01-IM5","P-0006763-T01-IM5","P-0006864-T01-IM5","P-0006978-T01-IM5",
"P-0007315-T01-IM5","P-0007631-T01-IM5","P-0008534-T01-IM5","P-0008584-T01-IM5",
"P-0009341-T01-IM5","P-0010049-T01-IM5","P-0010134-T01-IM5","P-0010364-T01-IM5",
"P-0010393-T01-IM5","P-0011110-T01-IM5","P-0011296-T01-IM5","P-0011455-T01-IM5",
"P-0012281-T01-IM5","P-0013087-T01-IM5","P-0013096-T01-IM5","P-0013285-T01-IM5",
"P-0013354-T01-IM5","P-0013820-T01-IM5","P-0014119-T01-IM5"
]

payload = {"sampleIds": sample_ids}

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

r = requests.post(
    f"{base_url}/molecular-profiles/{profile}/mutations/fetch",
    json=payload,
    headers=headers
)

mut = pd.DataFrame(r.json())

mut.to_csv("all_mutations_crc_msk2017_samples.csv", index=False)


# repeat if there are >1 studies, and merge into a single mut file

In [15]:
import pandas as pd
from pyfaidx import Fasta

# -----------------------------
# Inputs
# -----------------------------
mut = pd.read_csv("all_mutations_crc_msk2017_samples.csv")

# Change this path if needed
fasta_path = "references/hg19/hg19.fa"
ref = Fasta(fasta_path)

# -----------------------------
# Basic cleanup / added columns
# -----------------------------
mut = mut.copy()

# cBioPortal export uses 'keyword' as gene symbol here
mut["gene"] = mut["keyword"]

# HGVSc is not present in this export
mut["HGVSc"] = pd.NA

# Mark the specific target splice-creating mutation:
# chr5:112151184 A>G
mut["target splice"] = (
    mut["chr"].astype(str).str.replace(r"\.0$", "", regex=True).isin(["5", "chr5"]) &
    pd.to_numeric(mut["startPosition"], errors="coerce").eq(112151184) &
    mut["referenceAllele"].astype(str).str.upper().eq("A") &
    mut["variantAllele"].astype(str).str.upper().eq("G")
).astype(int)

# -----------------------------
# Keep all single-base substitutions
# This includes splice SNVs and splice-creating SNVs
# -----------------------------
is_sbs = (
    mut["referenceAllele"].astype(str).str.len().eq(1) &
    mut["variantAllele"].astype(str).str.len().eq(1) &
    mut["referenceAllele"].astype(str).str.upper().isin(list("ACGT")) &
    mut["variantAllele"].astype(str).str.upper().isin(list("ACGT"))
)

snv = mut.loc[is_sbs].copy()

# -----------------------------
# Helpers for COSMIC96 assignment
# -----------------------------
comp = str.maketrans("ACGT", "TGCA")

def revcomp(seq):
    return seq.translate(comp)[::-1]

def clean_chrom(chrom):
    chrom = str(chrom).strip()
    if chrom.endswith(".0"):
        chrom = chrom[:-2]
    return chrom

def get_context_pyfaidx(ref, chrom, pos1):
    chrom = clean_chrom(chrom)

    candidates = [chrom]
    if chrom.startswith("chr"):
        candidates.append(chrom.replace("chr", "", 1))
    else:
        candidates.append("chr" + chrom)

    used_chrom = None
    for c in candidates:
        if c in ref.keys():
            used_chrom = c
            break

    if used_chrom is None:
        raise KeyError(f"Chromosome {chrom} not found in FASTA")

    pos1 = int(pos1)

    # pyfaidx uses 0-based slicing, end-exclusive
    context = ref[used_chrom][pos1 - 2 : pos1 + 1].seq.upper()

    if len(context) != 3:
        return None, used_chrom

    return context, used_chrom

def to_cosmic96(context, ref_base, alt_base):
    context = context.upper()
    ref_base = ref_base.upper()
    alt_base = alt_base.upper()

    if len(context) != 3:
        return None

    if context[1] != ref_base:
        return None

    # Normalize to pyrimidine representation
    if ref_base in ["A", "G"]:
        context = revcomp(context)
        ref_base = revcomp(ref_base)
        alt_base = revcomp(alt_base)

    if ref_base not in ["C", "T"]:
        return None

    return f"{context[0]}[{ref_base}>{alt_base}]{context[2]}"

# -----------------------------
# Annotate trinucleotide context and COSMIC96
# -----------------------------
contexts = []
classes = []
problems = []
used_chrs = []

for _, row in snv.iterrows():
    try:
        context, used_chr = get_context_pyfaidx(ref, row["chr"], row["startPosition"])
        ref_base = str(row["referenceAllele"]).upper()
        alt_base = str(row["variantAllele"]).upper()

        cosmic = to_cosmic96(context, ref_base, alt_base)

        contexts.append(context)
        classes.append(cosmic)
        used_chrs.append(used_chr)

        if context is None:
            problems.append("Could not extract 3bp context")
        elif cosmic is None:
            problems.append(
                f"Reference allele mismatch: FASTA middle base={context[1]}, mutation ref={ref_base}"
            )
        else:
            problems.append(None)

    except Exception as e:
        contexts.append(None)
        classes.append(None)
        used_chrs.append(None)
        problems.append(str(e))

snv["used_chr"] = used_chrs
snv["trinuc_context"] = contexts
snv["COSMIC96"] = classes
snv["problem"] = problems

# -----------------------------
# Build final dataframe
# -----------------------------
final_df = snv[[
    "sampleId",
    "patientId",
    "gene",
    "HGVSc",
    "target splice",
    "chr",
    "used_chr",
    "startPosition",
    "endPosition",
    "referenceAllele",
    "variantAllele",
    "proteinChange",
    "mutationType",
    "variantType",
    "ncbiBuild",
    "trinuc_context",
    "COSMIC96",
    "problem"
]].copy()

# -----------------------------
# Save outputs
# -----------------------------
final_df.to_csv("all_mutations_crc_msk2017_samples_cosmic96.csv", index=False)

counts = final_df["COSMIC96"].value_counts(dropna=True).sort_index()
counts.to_csv("cosmic96_counts.csv", header=["count"])

# -----------------------------
# Checks
# -----------------------------
print(final_df.head(20))

print("\nTarget splice rows:")
print(
    final_df.loc[final_df["target splice"] == 1, [
        "sampleId",
        "gene",
        "proteinChange",
        "mutationType",
        "chr",
        "startPosition",
        "referenceAllele",
        "variantAllele",
        "trinuc_context",
        "COSMIC96",
        "problem"
    ]]
)


             sampleId  patientId                   gene HGVSc  target splice  \
0   P-0003365-T01-IM5  P-0003365         APC truncating  <NA>              1   
2   P-0003365-T01-IM5  P-0003365    ASXL1 V555 missense  <NA>              0   
4   P-0003575-T01-IM5  P-0003575         APC truncating  <NA>              1   
5   P-0003575-T01-IM5  P-0003575         KDR truncating  <NA>              0   
6   P-0003575-T01-IM5  P-0003575     MDM2 R189 missense  <NA>              0   
7   P-0003575-T01-IM5  P-0003575     TP53 M133 missense  <NA>              0   
8   P-0003710-T01-IM5  P-0003710         APC truncating  <NA>              1   
9   P-0003710-T01-IM5  P-0003710   PTPRS A1758 missense  <NA>              0   
11  P-0003710-T01-IM5  P-0003710   PTPRT E1124 missense  <NA>              0   
12  P-0003710-T01-IM5  P-0003710    TET2 A1768 missense  <NA>              0   
13  P-0004159-T01-IM5  P-0004159     ALOX12B truncating  <NA>              0   
14  P-0004159-T01-IM5  P-0004159        